# FreshBasket churn: data quality and EDA

Run `python src\churn_analysis.py` from the project root first. This notebook reviews the source tables and generated analysis outputs. The production pipeline restricts model features to activity through March 2024; the April-June label window is used only for the target.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
WORKBOOK = ROOT / 'FreshBasket_Loyalty_Churn_Dataset.xlsx'
OUTPUTS = ROOT / 'outputs'
REPORTS = ROOT / 'reports'
assert WORKBOOK.exists(), f'Workbook not found: {WORKBOOK}'
assert (OUTPUTS / 'customer_modeling_dataset.csv').exists(), 'Run src/churn_analysis.py first.'
profile = pd.read_excel(WORKBOOK, sheet_name='fb Customer Profile', header=1)
activity = pd.read_excel(WORKBOOK, sheet_name='fb Monthly Activity', header=1)
labels = pd.read_excel(WORKBOOK, sheet_name='fb Churn Label', header=1)
quality = pd.read_csv(OUTPUTS / 'data_quality_summary.csv')
customers = pd.read_csv(OUTPUTS / 'customer_modeling_dataset.csv')
metrics = pd.read_csv(OUTPUTS / 'model_comparison.csv')
segments = pd.read_csv(OUTPUTS / 'segment_churn_summary.csv')
print('Profile:', profile.shape, '| Activity:', activity.shape, '| Labels:', labels.shape)
print('Customer-level dataset:', customers.shape)

## Source data and target prevalence

The source data is sparse by customer-month. Read the quality table before interpreting missing months as zero activity.

In [ ]:
display(quality)
display(labels['CHURNED'].value_counts(dropna=False).rename_axis('CHURNED').to_frame('customers'))
print('Raw duplicate rows — profile:', profile.duplicated().sum(),
      'activity:', activity.duplicated().sum(), 'labels:', labels.duplicated().sum())
activity['MONTH'] = pd.to_datetime(activity['MONTH'])
print('Activity date range:', activity['MONTH'].min(), 'to', activity['MONTH'].max())
print('Negative spend:', (activity['TOTAL_SPEND'] < 0).sum(),
      '| zero transactions:', (activity['TRANSACTIONS'] == 0).sum())

## Monthly coverage and activity

Rows represent recorded activity, not necessarily every active loyalty member in each month.

In [ ]:
monthly_rows = activity.groupby(activity['MONTH'].dt.to_period('M')).size()
monthly_customers = activity.groupby(activity['MONTH'].dt.to_period('M'))['CUSTOMER_ID'].nunique()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
monthly_rows.plot(ax=axes[0], marker='o', title='Recorded activity rows per month')
monthly_customers.plot(ax=axes[1], marker='o', title='Unique customers with recorded activity')
for ax in axes:
    ax.set_xlabel('Month')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

## Customer-level churn patterns

The following charts use customers with sufficient history and exclude rows flagged as label-definition conflicts from evaluation.

In [ ]:
eligible = customers.loc[
    customers['CHURNED'].notna()
    & customers['INSUFFICIENT_HISTORY_FLAG'].eq(0)
    & ~customers['_label_mismatch'].fillna(True)
].copy()
display(metrics)
tier = segments.loc[segments['segment_type'].eq('MEMBERSHIP_TIER')].sort_values('segment')
display(tier)
fig, ax = plt.subplots(figsize=(7, 4))
tier.set_index('segment')[['observed_churn_rate', 'mean_predicted_risk']].plot(
    kind='bar', ax=ax, ylim=(0, 1), title='Observed churn and model risk by tier'
)
ax.set_ylabel('Rate')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
eligible.boxplot(column='recency_months', by='CHURNED', ax=axes[0])
axes[0].set_title('Pre-label purchase recency by churn label')
axes[0].set_xlabel('CHURNED')
eligible.boxplot(column='transactions_recent_vs_previous', by='CHURNED', ax=axes[1])
axes[1].set_title('Change in transactions by churn label')
axes[1].set_xlabel('CHURNED')
plt.suptitle('')
plt.tight_layout()
importance = pd.read_csv(OUTPUTS / 'feature_importance.csv').head(12)
display(importance)

## Interpretation cautions

- The observed associations and permutation importance are not causal explanations.
- The cohort holdout separates signup cohorts but shares the same outcome period; it is not prospective future-time validation.
- Incomplete customer-month capture can bias activity and recency features.
- Review `reports/final_report.md` and `outputs/retention_scenario_summary.json` for limitations and the non-causal coupon scenario.